# FMP Indexes

Read-first examples for FMP-backed index OHLCV data.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openbb import obb

for env_dir in [Path.cwd(), *Path.cwd().parents]:
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

INDEX_SYMBOL = "^GSPC"

In [2]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

In [3]:
index_routes = {INDEX_PRICES_SECTION: INDEX_PRICES_LIBRARY}
route_table(index_routes)

,section,openbb_route,provider_library
0,index_prices,index.price.historical,fmp_index_prices


In [4]:
pd.Series(DEFAULT_INDEX_SYMBOLS, name="default_index_symbols")

0    ^GSPC
1     ^DJI
2    ^IXIC
3     ^VIX
4     ^RUT
Name: default_index_symbols, dtype: object

In [5]:
if RUN_REFRESH:
    warehouse.market_prices.refresh(INDEX_SYMBOL, section=INDEX_PRICES_SECTION, provider=PROVIDER)

state_table(INDEX_SYMBOL, [INDEX_PRICES_SECTION])

,symbol,section,provider,min_date,max_date,row_count,column_count,columns_present,last_fetched_at
0,^GSPC,index_prices,fmp,2006-08-03,2026-06-18,5000,8,"(close, fmp__change, fmp__change_percent, fmp_...",2026-06-20T16:10:05.150069+00:00


In [6]:
preview(warehouse.market_prices.read(INDEX_SYMBOL, section=INDEX_PRICES_SECTION, provider=PROVIDER))

,open,high,low,close,volume,fmp__vwap,fmp__change,fmp__change_percent
date,,,,,,,,
2026-06-12,7410.85,7456.40,7363.01,7431.45,4950530000,7415.4275,20.60,0.002780
2026-06-15,7516.75,7577.92,7516.75,7554.28,5674670000,7541.4250,37.53,0.004993
2026-06-16,7548.78,7564.96,7508.68,7511.34,5286210000,7533.4400,-37.44,-0.004960
2026-06-17,7524.50,7532.17,7402.61,7420.11,5883740000,7469.8475,-104.39,-0.013900
2026-06-18,7487.36,7511.07,7468.32,7500.57,9061110000,7491.8300,13.21,0.001764


<!-- quant-trader-eda -->
## Quant Trader EDA

Index checks: market regime, drawdown, realized volatility, and trend state for benchmark timing/risk overlays.

In [7]:
def price_eda(frame: pd.DataFrame, annualization: int = 252) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame is None or frame.empty or "close" not in frame.columns:
        return pd.DataFrame(), pd.DataFrame()
    close = pd.to_numeric(frame["close"], errors="coerce").dropna()
    returns = close.pct_change().dropna()
    if returns.empty:
        return pd.DataFrame(), pd.DataFrame()
    equity_curve = (1 + returns).cumprod()
    drawdown = equity_curve / equity_curve.cummax() - 1
    summary = pd.DataFrame(
        {
            "observations": [int(len(returns))],
            "start": [close.index.min()],
            "end": [close.index.max()],
            "total_return": [close.iloc[-1] / close.iloc[0] - 1],
            "annualized_return": [(1 + returns.mean()) ** annualization - 1],
            "annualized_volatility": [returns.std() * annualization ** 0.5],
            "sharpe_0_rf": [returns.mean() / returns.std() * annualization ** 0.5 if returns.std() else None],
            "max_drawdown": [drawdown.min()],
            "best_day": [returns.max()],
            "worst_day": [returns.min()],
        }
    )
    diagnostics = pd.DataFrame(
        {
            "close": close,
            "return": returns.reindex(close.index),
            "rolling_21d_vol": returns.rolling(21).std() * annualization ** 0.5,
            "rolling_63d_vol": returns.rolling(63).std() * annualization ** 0.5,
            "drawdown": drawdown.reindex(close.index),
        }
    )
    return summary, diagnostics

In [8]:
index_prices = warehouse.market_prices.read(INDEX_SYMBOL, section=INDEX_PRICES_SECTION, provider=PROVIDER)
index_summary, index_diagnostics = price_eda(index_prices)
index_summary

,observations,start,end,total_return,annualized_return,annualized_volatility,sharpe_0_rf,max_drawdown,best_day,worst_day
0,4999,2006-08-03,2026-06-18,4.858585,0.11439,0.195712,0.553521,-0.567754,0.1158,-0.119841


In [9]:
preview(index_diagnostics[["close", "rolling_21d_vol", "rolling_63d_vol", "drawdown"]], rows=10)

,close,rolling_21d_vol,rolling_63d_vol,drawdown
date,,,,
2026-06-05,7383.73,0.131344,0.150770,-0.029704
2026-06-08,7405.72,0.130870,0.150168,-0.026814
2026-06-09,7386.66,0.127966,0.150213,-0.029319
2026-06-10,7267.00,0.139353,0.154232,-0.045043
2026-06-11,7394.31,0.153207,0.153996,-0.028314
2026-06-12,7431.45,0.152836,0.153328,-0.023433
2026-06-15,7554.28,0.161319,0.155254,-0.007292
2026-06-16,7511.34,0.156245,0.155998,-0.012935
2026-06-17,7420.11,0.162445,0.155438,-0.024923


In [10]:
if index_prices.empty or "close" not in index_prices.columns:
    pd.DataFrame()
else:
    close = pd.to_numeric(index_prices["close"], errors="coerce").dropna()
    regime = pd.DataFrame({
        "close": close,
        "sma_50": close.rolling(50).mean(),
        "sma_200": close.rolling(200).mean(),
        "above_200d_sma": close > close.rolling(200).mean(),
        "distance_to_200d_sma": close / close.rolling(200).mean() - 1,
    })
    preview(regime, rows=10)

<!-- quant-trader-signal-eda -->
## Signal Research for P&L

Index EDA is useful for risk overlays: trend filters, volatility targeting, drawdown control, and risk-on/risk-off regime flags.

In [11]:
def signal_backtest(close: pd.Series, signal: pd.Series, annualization: int = 252) -> pd.DataFrame:
    close = pd.to_numeric(close, errors="coerce").dropna()
    returns = close.pct_change().dropna()
    aligned_signal = signal.reindex(returns.index).shift(1).fillna(0).clip(-1, 1)
    strategy_returns = aligned_signal * returns
    if strategy_returns.empty:
        return pd.DataFrame()
    curve = (1 + strategy_returns).cumprod()
    buy_hold = (1 + returns).cumprod()
    drawdown = curve / curve.cummax() - 1
    turnover = aligned_signal.diff().abs().fillna(aligned_signal.abs())
    return pd.DataFrame({
        "strategy_total_return": [curve.iloc[-1] - 1],
        "buy_hold_total_return": [buy_hold.iloc[-1] - 1],
        "strategy_annualized_return": [(1 + strategy_returns.mean()) ** annualization - 1],
        "strategy_annualized_volatility": [strategy_returns.std() * annualization ** 0.5],
        "strategy_sharpe_0_rf": [strategy_returns.mean() / strategy_returns.std() * annualization ** 0.5 if strategy_returns.std() else None],
        "strategy_max_drawdown": [drawdown.min()],
        "hit_rate": [(strategy_returns > 0).mean()],
        "average_exposure": [aligned_signal.abs().mean()],
        "average_daily_turnover": [turnover.mean()],
        "latest_signal": [aligned_signal.iloc[-1]],
    })

In [12]:
index_prices = warehouse.market_prices.read(INDEX_SYMBOL, section=INDEX_PRICES_SECTION, provider=PROVIDER)
if index_prices.empty or "close" not in index_prices.columns:
    pd.DataFrame()
else:
    close = index_prices["close"]
    trend_signal = (close > close.rolling(200).mean()).astype(float)
    signal_backtest(close, trend_signal)

In [13]:
if index_prices.empty or "close" not in index_prices.columns:
    pd.DataFrame()
else:
    close = index_prices["close"]
    returns = close.pct_change()
    vol_21 = returns.rolling(21).std() * 252 ** 0.5
    regime = pd.DataFrame({
        "close": close,
        "above_200d_sma": close > close.rolling(200).mean(),
        "momentum_63d": close.pct_change(63),
        "momentum_252d": close.pct_change(252),
        "realized_vol_21d": vol_21,
        "vol_target_weight_10pct_cap_1x": (0.10 / vol_21).clip(0, 1),
        "drawdown": close / close.cummax() - 1,
    })
    preview(regime, rows=15)

<!-- output-analysis -->
## Analysis Notes From This Run

For `^GSPC`, the stored FMP index series is above its 200-day average by about 8.7%, with latest 21-day realized volatility around 16.4%. The full-sample max drawdown is about 56.8%, while the simple trend-filter diagnostic materially reduces drawdown in this run.

The index notebook is most useful as a market-regime overlay. A quant trader can use this as a gating signal for gross exposure, volatility targets, or whether single-name signals should be allowed to run at full size.